In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import polars as pl
from influxdb_client  import InfluxDBClient , WriteOptions
import yaml
import requests


current_file = Path(globals().get('__vsc_ipynb_file__', None))
parent_dir = current_file.parent.parent
os.chdir(parent_dir)
print("Changed working directory to:", Path.cwd())

from dotenv import load_dotenv
load_dotenv("mlops_services/.env.mlops")

Changed working directory to: /home/legacy/Projects/Weather_forecasting_with_MLOps


True

In [8]:
from mlops_services.src.utils import  logs
logger = logs.setup_logger("test")
logger.info("Hello")

logger2 = logs.setup_logger("test2")
logger2.info("Hellooo")


2025-08-24 07:10:48,324 - INFO - Hello
2025-08-24 07:10:48,327 - INFO - Hellooo


In [10]:
from mlops_services.src.pipelines.train.a_retrieve import DataRetrieval

In [9]:
from mlops_services.src.utils.time_utils import convert_to_utc,include_time

# convert_to_utc('2023-01-01 00:00','2025-06-30 23:00', tz="Asia/Kolkata",ts_format="%Y-%m-%d %H:%M")

In [24]:
%run mlops_services/src/pipelines/train/a_retrieve.py local BLR "2023-01-01 00:00" "2025-06-30 23:00"

2025-08-24 08:30:52,275 - INFO - 01_Data_Retrieval --> Started
2025-08-24 08:30:52,286 - INFO - City : BLR
2025-08-24 08:30:52,288 - INFO - Trying to load historic data from local
2025-08-24 08:30:52,337 - INFO - Successfully loaded local data
2025-08-24 08:30:52,386 - INFO - Data saved in : mlops_services/data/raw/complete_set.parquet
2025-08-24 08:30:52,388 - INFO - 01_Data_Retrieval --> Completed


In [21]:
%run mlops_services/src/pipelines/train/b_split.py

2025-08-24 07:56:24,421 - INFO - 02_Data_Split --> Started
2025-08-24 07:56:24,511 - INFO - 02_Data_Split --> Completed


In [14]:
%run mlops_services/src/pipelines/train/c_preprocess.py

2025-08-27 11:41:21,867 - INFO - >>Preprocessing : train
2025-08-27 11:41:21,872 - INFO - Columns feteched : ['time', 'temperature_2m', 'cloud_cover', 'dew_point_2m', 'et0_fao_evapotranspiration', 'generationtime_ms', 'rain', 'relative_humidity_2m', 'surface_pressure', 'vapour_pressure_deficit', 'weather_code', 'wind_direction_10m', 'wind_gusts_10m', 'wind_speed_10m']
2025-08-27 11:41:21,877 - INFO - Dropped Nulls -- Count before:16416 & Count After:16416
2025-08-27 11:41:21,891 - INFO - Data resampled -- Frequency:1h Count before:16416 & Count After:16416
2025-08-27 11:41:21,896 - INFO - Features imputed -- Strategy:ffill                         
 Count before:{'time': [0], 'temperature_2m': [0], 'cloud_cover': [0], 'dew_point_2m': [0], 'et0_fao_evapotranspiration': [0], 'generationtime_ms': [0], 'rain': [0], 'relative_humidity_2m': [0], 'surface_pressure': [0], 'vapour_pressure_deficit': [0], 'weather_code': [0], 'wind_direction_10m': [0], 'wind_gusts_10m': [0], 'wind_speed_10m': [0]

In [8]:
%run mlops_services/src/pipelines/train/d_features.py

In [ ]:
%tb

In [12]:
df_train = pl.read_parquet('mlops_services/data/processed/train.parquet')
df_train

time,cloud_cover,dew_point_2m,et0_fao_evapotranspiration,generationtime_ms,rain,relative_humidity_2m,surface_pressure,vapour_pressure_deficit,weather_code,wind_direction_10m,wind_gusts_10m,wind_speed_10m
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2022-12-31 18:00:00 UTC,1.0,11.7,0.02,8.804321,0.0,74.0,914.8,0.47,0.0,97.0,15.1,9.4
2022-12-31 19:00:00 UTC,0.0,11.6,0.02,8.804321,0.0,75.0,914.1,0.46,0.0,96.0,16.2,9.8
2022-12-31 20:00:00 UTC,83.0,12.1,0.01,8.804321,0.0,80.0,913.3,0.36,3.0,96.0,16.2,9.8
2022-12-31 21:00:00 UTC,78.0,12.7,0.0,8.804321,0.0,84.0,913.0,0.27,2.0,100.0,16.9,10.2
2022-12-31 22:00:00 UTC,61.0,13.1,0.0,8.804321,0.0,89.0,912.9,0.18,2.0,99.0,16.9,9.1
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-11-14 13:00:00 UTC,100.0,19.9,0.0,8.804321,0.0,90.0,912.8,0.25,3.0,77.0,22.0,10.0
2024-11-14 14:00:00 UTC,100.0,20.1,0.0,8.804321,0.0,93.0,913.4,0.17,3.0,77.0,22.0,10.5
2024-11-14 15:00:00 UTC,100.0,20.0,0.0,8.804321,0.0,93.0,913.7,0.16,3.0,78.0,26.6,10.3


In [13]:
df_train.columns

['time',
 'cloud_cover',
 'dew_point_2m',
 'et0_fao_evapotranspiration',
 'generationtime_ms',
 'rain',
 'relative_humidity_2m',
 'surface_pressure',
 'vapour_pressure_deficit',
 'weather_code',
 'wind_direction_10m',
 'wind_gusts_10m',
 'wind_speed_10m']

In [ ]:
df_train.count().to_dict(as_series=False)

In [10]:
from mlops_services.src.utils import command_line, configs
config = configs.Config.from_yaml(
            "mlops_services/config/model_params.yaml"
        )


In [21]:
'jj' in config.algorithms.__dict__.keys()

False